# One T4x2 Experiment - Smoke First, Full Later

This notebook runs one YOLO11s segmentation experiment on both Kaggle T4 GPUs with live logging, prints results inside the notebook, and packages the outputs at the end. Start with `RUN_STAGE = "smoke"`; after it passes, change only that variable to `"full"` and run again.

In [ ]:
from pathlib import Path

# Change this from "smoke" to "full" only after the smoke test passes.
RUN_STAGE = "smoke"  # "smoke" or "full"

# One experiment only. Change this later when E01 is fully done.
EXPERIMENT_CONFIG = "configs/experiments/e01_source_rgb_yolo11s.yaml"
EXPERIMENT_NAME = "E01_source_rgb_yolo11s"

# T4x2 defaults. Batch 16 is split across both GPUs by Ultralytics/DDP.
YOLO_DEVICE = "0,1"
YOLO_BATCH_SMOKE = "16"
YOLO_BATCH_FULL = "16"
YOLO_WORKERS = "2"
YOLO_PATIENCE = "25"
YOLO_RESUME = "auto"

# Leave ZIP_PATH as None to auto-detect the package under /kaggle/input.
ZIP_PATH = None

WORK_DIR = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content")
REPO_DIR = WORK_DIR / "domain-adaptation-segmentation"
RUNS_BASE = WORK_DIR / "runs"
OUTPUT_ROOT = RUNS_BASE / f"kaggle_single_e01_{RUN_STAGE}"
REPORT_DIR = Path(f"reports/tables/kaggle_single_e01_{RUN_STAGE}")
YOLO_EPOCHS = "1" if RUN_STAGE == "smoke" else "100"
YOLO_BATCH = YOLO_BATCH_SMOKE if RUN_STAGE == "smoke" else YOLO_BATCH_FULL

assert RUN_STAGE in {"smoke", "full"}, RUN_STAGE
print("RUN_STAGE:", RUN_STAGE)
print("EXPERIMENT:", EXPERIMENT_CONFIG)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("YOLO_DEVICE/BATCH/EPOCHS:", YOLO_DEVICE, YOLO_BATCH, YOLO_EPOCHS)


## Helpers

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import datetime

def stage(title):
    print("\n" + "=" * 90)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {title}")
    print("=" * 90)

def run_live(command, cwd=None, env=None):
    stage("RUN: " + " ".join(map(str, command)))
    process = subprocess.Popen(
        list(map(str, command)),
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    print(f"\n[return_code] {return_code}")
    if return_code != 0:
        raise RuntimeError(f"Command failed with return code {return_code}: {command}")

def show_tail(path, lines=40):
    path = Path(path)
    stage(f"TAIL: {path}")
    if not path.exists():
        print("missing")
        return
    text = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(text[-lines:]))

def running_on_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False

def running_on_kaggle():
    return Path("/kaggle/input").exists()


## Stage 1 - Extract Package

In [ ]:
stage("Stage 1 - Extract package")

if ZIP_PATH is None:
    candidates = sorted(Path("/kaggle/input").glob("**/domain-adaptation-segmentation-kaggle.zip")) if running_on_kaggle() else []
    if not candidates:
        raise FileNotFoundError(
            "Could not auto-detect domain-adaptation-segmentation-kaggle.zip. "
            "Set ZIP_PATH manually to the uploaded package path."
        )
    ZIP_PATH = str(candidates[0])

print("ZIP_PATH:", ZIP_PATH)
if not (REPO_DIR / "src/domain_adaptation_segmentation").exists():
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(REPO_DIR)
else:
    print("Repository already extracted; reusing:", REPO_DIR)

required = [
    REPO_DIR / "requirements.txt",
    REPO_DIR / "src/domain_adaptation_segmentation",
    REPO_DIR / "data/manifests/dataset_yamls/source_rgb.yaml",
]
for path in required:
    print(path, "OK" if path.exists() else "MISSING")
    if not path.exists():
        raise FileNotFoundError(path)


## Stage 2 - Install Dependencies And Check T4x2x2

In [ ]:
stage("Stage 2 - Install dependencies")
run_live([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO_DIR)

stage("Stage 2 - GPU check")
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. Enable T4x2 GPU before running training.")


## Stage 3 - Run Smoke Or Full Experiment With Live Logs

In [ ]:
stage("Stage 3 - Training setup")
env = os.environ.copy()
env.update({
    "PYTHONPATH": str(REPO_DIR / "src"),
    "OUTPUT_ROOT": str(OUTPUT_ROOT),
    "REPORT_DIR": str(REPORT_DIR),
    "EXPERIMENT_CONFIG": EXPERIMENT_CONFIG,
    "YOLO_DEVICE": YOLO_DEVICE,
    "YOLO_EPOCHS": YOLO_EPOCHS,
    "YOLO_BATCH": YOLO_BATCH,
    "YOLO_WORKERS": YOLO_WORKERS,
    "YOLO_PATIENCE": YOLO_PATIENCE,
    "YOLO_RESUME": YOLO_RESUME,
})
for key in ["OUTPUT_ROOT", "REPORT_DIR", "EXPERIMENT_CONFIG", "YOLO_DEVICE", "YOLO_EPOCHS", "YOLO_BATCH", "YOLO_WORKERS", "YOLO_RESUME"]:
    print(key, "=", env[key])

run_live(["bash", "scripts/remote/kaggle_run_one_experiment.sh"], cwd=REPO_DIR, env=env)

run_dir = OUTPUT_ROOT / "experiments" / EXPERIMENT_NAME
show_tail(run_dir / "stdout.log", lines=80)


## Stage 4 - Print Results In Notebook

In [ ]:
stage("Stage 4 - Results")
import pandas as pd

run_dir = OUTPUT_ROOT / "experiments" / EXPERIMENT_NAME
status_path = run_dir / "status.json"
results_path = run_dir / "results.csv"
summary_path = REPO_DIR / REPORT_DIR / "summary_results.csv"

print("run_dir:", run_dir)
if status_path.exists():
    status = json.loads(status_path.read_text(encoding="utf-8"))
    print("status:", status.get("status"))
    print("started:", status.get("started_at_utc"))
    print("finished:", status.get("finished_at_utc"))
    print("elapsed_seconds:", status.get("elapsed_seconds"))
    print("resume_checkpoint:", status.get("resume_checkpoint"))
else:
    print("status.json missing")

if results_path.exists():
    df = pd.read_csv(results_path)
    df.columns = [column.strip() for column in df.columns]
    display(df.tail())
    last = df.iloc[-1]
    metric_cols = [
        "epoch",
        "metrics/mAP50(M)",
        "metrics/mAP50-95(M)",
        "metrics/precision(M)",
        "metrics/recall(M)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)",
    ]
    print("\nLatest key metrics:")
    for col in metric_cols:
        if col in df.columns:
            print(f"{col}: {last[col]}")
else:
    print("results.csv missing")

if summary_path.exists():
    print("\nSummary table:")
    display(pd.read_csv(summary_path))


## Stage 5 - Store And Download Results

In [ ]:
stage("Stage 5 - Package outputs")
bundle_dir = WORK_DIR / f"{RUN_STAGE}_e01_artifacts"
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)

if OUTPUT_ROOT.exists():
    shutil.copytree(OUTPUT_ROOT, bundle_dir / OUTPUT_ROOT.name)
if (REPO_DIR / REPORT_DIR).exists():
    shutil.copytree(REPO_DIR / REPORT_DIR, bundle_dir / "tables")

archive_base = WORK_DIR / f"{RUN_STAGE}_e01_results"
archive_path = shutil.make_archive(str(archive_base), "zip", bundle_dir)
print("Result zip:", archive_path)

if running_on_colab():
    from google.colab import files
    print("Starting browser download via google.colab.files.download...")
    files.download(archive_path)
else:
    print("Kaggle note: automatic local download is not available from the kernel.")
    print("Download this file from the notebook output/files panel:")
    print(archive_path)
    try:
        from IPython.display import FileLink, display
        display(FileLink(archive_path))
    except Exception as exc:
        print("Could not render FileLink:", exc)
